In [1]:
from src.utils import load_config
from src.models.personalized_ad_frequency import MODEL_REGISTRY
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler

config = load_config("configs/personalized_ad_frequency.yaml")


In [2]:
processed_data_path = config['processed_data_path']
if not os.path.exists(processed_data_path):
    print(f"❌ HATA: Gerekli bir dosya bulunamadı -> {processed_data_path}")
        

In [3]:
df = pd.read_excel(processed_data_path)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3923 entries, 0 to 3922
Data columns (total 11 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   user_key_id                        3923 non-null   object 
 1   country                            3923 non-null   object 
 2   age                                3923 non-null   int64  
 3   sex                                3923 non-null   object 
 4   easy_completion_rate               3923 non-null   int64  
 5   medium_completion_rate             3923 non-null   int64  
 6   hard_completion_rate               3923 non-null   float64
 7   session_count                      3923 non-null   int64  
 8   total_session_time                 3923 non-null   int64  
 9   total_ad_viewed                    3923 non-null   int64  
 10  total_earned_reward_amount_in_usd  3923 non-null   float64
dtypes: float64(2), int64(6), object(3)
memory usage: 337.3+ 

In [9]:
import mlflow
import mlflow.sklearn
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# Make sure this path exists or create it
output_dir = "./mlruns/ad_frequency"
os.makedirs(output_dir, exist_ok=True)

label_encoders = {}
for col in ['country', 'sex']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = list(le.classes_)

# Scale features once
features = df[['total_ad_viewed', 'session_count', 'total_session_time', 'total_earned_reward_amount_in_usd']]
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

mlflow.set_experiment(config['experiment_name'])

for model_config in config['models']:
    model_name = model_config['model_name']
    ModelClass = MODEL_REGISTRY[model_name]

    for params in model_config['param_grid']:
        with mlflow.start_run(run_name=f"{model_name}"):

            # Log model name and parameters
            mlflow.log_param("model_name", model_name)
            mlflow.log_params(params)

            model = ModelClass(params)

            # Special handling for GaussianMixture
            if model_name == "GaussianMixtureClusterer":
                model.model.fit(features_scaled)
                df['cluster'] = model.model.predict(features_scaled)
            else:
                df['cluster'] = model.fit_predict(features_scaled)

            cluster_label_map = {
                0: 'Medium',
                1: 'Frequent',
                2: 'Rare',
                -1: 'Rare'  # for DBSCAN
            }
            df['user_ad_frequency_category'] = df['cluster'].map(cluster_label_map)

            # Save result to CSV
            result_path = os.path.join(output_dir, f"{model_name}_{str(params)}_results.csv".replace(" ", "_").replace(":", ""))
            df[['user_key_id', 'total_ad_viewed', 'cluster', 'user_ad_frequency_category']].to_csv(result_path, index=False)
            mlflow.log_artifact(result_path)

            labels = df['cluster']
            num_clusters = labels.nunique()

            if num_clusters >= 3:
                try:
                    sil = silhouette_score(features_scaled, labels)
                    ch = calinski_harabasz_score(features_scaled, labels)
                    db = davies_bouldin_score(features_scaled, labels)
                    
                    mlflow.log_metric("silhouette_score", sil)
                    mlflow.log_metric("calinski_harabasz_score", ch)
                    mlflow.log_metric("davies_bouldin_score", db)
                except:
                    print(f"Metric logging failed: {e}")
                    
                # Log individual cluster sizes
                cluster_counts = df['cluster'].value_counts().to_dict()
                for cluster_id, count in cluster_counts.items():
                    mlflow.log_metric(f"cluster_{cluster_id}_size", count)

            try:
                mlflow.sklearn.log_model(model.model, name="model")

                mlflow.log_dict(label_encoders, "ad_frequency_label_encoders.json")

                run_info = {
                    "training_columns": list(features)
                }
                mlflow.log_dict(run_info, "ad_frequency_run_info.json")
            except Exception as e:
                print(f"Model logging failed: {e}")

            print(f"Logged {model_name} with params {params}")


Traceback (most recent call last):
  File "/Users/emirhanbaran/Desktop/ai-based-game-platform-v1/venv/lib/python3.13/site-packages/mlflow/store/tracking/file_store.py", line 347, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
  File "/Users/emirhanbaran/Desktop/ai-based-game-platform-v1/venv/lib/python3.13/site-packages/mlflow/store/tracking/file_store.py", line 445, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
  File "/Users/emirhanbaran/Desktop/ai-based-game-platform-v1/venv/lib/python3.13/site-packages/mlflow/store/tracking/file_store.py", line 1588, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
  File "/Users/emirhanbaran/Desktop/ai-based-game-platform-v1/venv/lib/python3.13/site-packages/mlflow/store/tracking/file_store.py", line 1581, in _read_helper
    result = read_yaml(root, file_name)
  File "/Users/emirhanbaran/Desktop/ai-based-game-platform-v1/venv/lib/python3

Logged DBSCANClusterer with params {'eps': 0.3, 'min_samples': 3, 'metric': 'euclidean', 'algorithm': 'auto'}


2025/06/20 00:10:03 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/06/20 00:10:03 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


Logged DBSCANClusterer with params {'eps': 0.4, 'min_samples': 3, 'metric': 'manhattan', 'algorithm': 'auto'}


2025/06/20 00:10:04 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/06/20 00:10:04 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


...

Logged AgglomerativeClusterer with params {'n_clusters': 3, 'metric': 'euclidean', 'linkage': 'ward'}


2025/06/20 00:10:21 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Logged MeanShiftClusterer with params {'bandwidth': 2, 'bin_seeding': True}


2025/06/20 00:10:23 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Logged GaussianMixtureClusterer with params {'n_components': 3, 'covariance_type': 'full', 'max_iter': 100}


In [43]:
cluster_summary = df.groupby('cluster')['total_ad_viewed'].mean().sort_values()

cluster_summary

cluster
0      1.989401
1     31.475634
2    273.093590
Name: total_ad_viewed, dtype: float64

In [44]:

cluster_label_map = {
     0: 'Rare',
     1: 'Medium',
     2: 'Frequent'
}

df['user_ad_frequency_category'] = df['cluster'].map(cluster_label_map)

df[['user_key_id', 'total_ad_viewed', 'cluster', 'user_ad_frequency_category']].head()


,user_key_id,total_ad_viewed,cluster,user_ad_frequency_category
0,bVas4SMsXQ,15,2,Frequent
1,Hm917zE2Mc,5,1,Medium
2,wFfzWC1Ydt,23,1,Medium
3,73S29MCQWv,3,0,Rare
4,B4A9tTZMAE,1,0,Rare


In [45]:
category_counts_df = df['user_ad_frequency_category'].value_counts().reset_index()
category_counts_df.columns = ['user_ad_frequency_category', 'user_count']
print(category_counts_df)

  user_ad_frequency_category  user_count
0                       Rare        1604
1                     Medium        1539
2                   Frequent         780
